# Advanced Parallel Image Processing

This notebook demonstrates MPI-based image processing with **Guided Scheduling** and multiple processing modes.

Implemented features:
- **Custom Modes**: Box Blur and Edge Detection.
- **Scheduling**: Dynamic load balancing (Guided approach) via Master-Worker pattern.
- **Communication**: Using explicit `MPI_Send` and `MPI_Recv` for task assignment.
- **Threads**: OpenMP integrated for CPU-side parallelization.

In [ ]:
import subprocess
import re
import numpy as np
import matplotlib.pyplot as plt
import os

In [ ]:
input_img = r"./images/input.jpg"
output_prefix = r"./images/output"
modes = ["blur", "edge"]
procs = [2, 4, 8, 16] # Use at least 2 for Master-Worker

In [ ]:
results = {}
for mode in modes:
    mode_times = []
    for p in procs:
        out_file = f"{output_prefix}_{mode}_{p}.jpg"
        # Using --oversubscribe for testing on limited CPU cores
        cmd = [
            "mpirun", "--oversubscribe",
            "-n", str(p),
            "./scripts/parallel_image",
            input_img,
            out_file,
            mode
        ]
        
        print(f"Running {mode} with {p} ranks...")
        res = subprocess.run(cmd, capture_output=True, text=True)
        
        # Parse time from Master's output
        t_match = re.search(r"Time taken: ([0-9.]+)", res.stdout)
        if t_match:
            mode_times.append(float(t_match.group(1)))
        else:
            print(f"Error parsing output for {p} ranks!")
            print(res.stdout)
    results[mode] = mode_times

## 1. Speedup Analysis (Guided Scheduling)

Guided scheduling starts with large chunks and tapers off, providing excellent load balancing for potentially non-uniform workloads.

In [ ]:
plt.figure(figsize=(10, 5))
for mode in modes:
    if mode in results and len(results[mode]) == len(procs):
        plt.plot(procs, results[mode], marker='o', label=f"{mode.capitalize()} Filter")

plt.xlabel("Number of Processors (including Master)")
plt.ylabel("Execution Time (seconds)")
plt.title("Image Processing Performance with Guided MPI Scheduling")
plt.legend()
plt.grid(True)
plt.show()

## 2. Result Verification

Check the `images/` folder to see the `output_blur_X.jpg` and `output_edge_X.jpg` results.